# ⏳ 05 — Shock Persistence and Event Studies (hourly resolution)

How long does a shock last, and does it depend on its sector and shock type (macro-systemic vs idiosyncratic)?

1. Event windows around historical crises (hourly cumulative log returns).
2. Persistence in hours: decay of deseasonalised returns $r_{\text{ds}}$ against matched non-shock nulls.
3. Sector decay half-lives & temporal stability (train, val, test joint block bootstrap).
4. Implications for dynamic contagion network models.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"

from cryptograph.analysis.event_study import event_window

hp = torch.load(f"{V2}/1h/panel.pt", weights_only=False)
times = pd.to_datetime(hp["times"])
model = hp["model_coins"]
close = pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin").reindex(index=times, columns=model)
r = np.log(close).diff()
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
events = events[events["date"] >= "2020-01-08"]

## 1. Event windows (hourly)

In [2]:
rows = []
for _, e in events.iterrows():
    w = event_window(r, e["date"], pre=24, post=72, freq="h")
    rows.append(pd.DataFrame({"hour": w.index, "median cumulative return": w.median(axis=1).values,
                              "event": f"{e['date'].date()} {e['name'][:28]}"}))
ew = pd.concat(rows)
fig = px.line(ew, x="hour", y="median cumulative return", color="event",
              title="Median cumulative log return across the 30 coins, 24 h before to 72 h after each event (hour 0 = 00:00 UTC of the event date)")
fig.add_vline(x=0, line_dash="dash", line_color="#ff5252")
fig.update_layout(height=460, yaxis_tickformat=".0%")
fig.show()
display(ew[ew["hour"].isin([0, 6, 24, 72])].pivot(index="event", columns="hour", values="median cumulative return")
        .map("{:+.1%}".format))

hour,0,6,24,72
event,,,,
2020-03-12 COVID Black Thursday,-0.7%,-9.3%,-53.3%,-42.3%
2021-05-19 China bans financial institu,-0.5%,-13.6%,-51.5%,-46.2%
2022-05-09 Terra UST depeg / LUNA colla,+1.3%,-2.4%,-15.6%,-32.2%
2022-06-13 Celsius withdrawal freeze /,-3.3%,-7.5%,-12.2%,+0.8%
2022-11-08 FTX collapse,-0.4%,-6.2%,-15.2%,-16.6%
2023-03-10 SVB failure / USDC depeg,-1.1%,-1.6%,+1.3%,+9.5%
2024-01-10 US spot BTC ETF approval,+0.5%,-0.2%,+9.0%,+4.1%
2024-08-05 Yen carry-trade unwind,-4.6%,-16.2%,-3.3%,-5.3%
2025-10-10 Oct-10 liquidation cascade,+0.1%,+0.6%,-20.6%,-15.2%


*What this shows:* the median coin's path around each crisis, hour by hour. Event dates are calendar days
(hour 0 is midnight UTC), so some moves start before hour 0; the exact hour of each shock is located in the
training phase from the shock labels. This is the starting point for measuring persistence in hours.

## 2. Persistence in hours (Deseasonalised Volatility Decay)

To measure volatility persistence without windowing artifacts, we avoid rolling-window volatility estimators (such as a 24h rolling window, which artificially prolongs elevated levels by 24h). Instead, we track the decay of deseasonalised absolute returns $|r_{\text{ds}}|$:

1. **Shock Definition**: Hours where $|z_{\text{ret}}| > \tau_{\text{train}}$ (`shock_today & shock_today_mask`) with a 24h refractory de-clustering filter.
2. **Shock Bar Exclusion**: We measure excess volatility strictly from $t+1$ to $t+72$, normalized by each coin's pre-event baseline (median $|r_{\text{ds}}|$ from $t-168$ to $t-24$):
   $$\text{Excess}(h) = \frac{\mathbb{E}[|r_{\text{ds}, t+h}|]}{\text{Baseline}}$$
3. **Matched Non-Shock Null**: For every shock episode, we draw a matched non-shock hour from the same coin and calendar month (with no trailing shock in the preceding 24h). The **excess gap** is:
   $$\text{Gap}(h) = \text{Excess}_{\text{shock}}(h) - \text{Excess}_{\text{null}}(h)$$
4. **Empirical Decay Half-Life**: The first hour $h \ge 1$ at which the excess gap falls to $\le 50\%$ of its initial value at $h=1$.
5. **Joint Calendar-Day Block Bootstrap**: 24h moving blocks drawn jointly across all coins (500 replicates), preserving cross-sectional dependence and volatility clustering.
6. **Sector Threshold**: Only sectors with $\ge 3$ coins from `data/external/sectors.csv` are included.

In [3]:
# Sector-level post-shock decay half-lives (all periods combined)
boot_df = pd.read_csv('../reports/persistence_sector_block_bootstrap.csv')
decay_all = boot_df[(boot_df['record_type'] == 'sector_decay_half_life') & (boot_df['split'] == 'all')].copy()

display_cols = [
    'sector', 'episode_count', 'estimate_hours', 'ci_95_low', 'ci_95_high',
    'gap_h1', 'gap_h24', 'gap_h72', 'p_value_beat_null'
]
disp = decay_all[display_cols].rename(columns={
    'sector': 'Sector (≥3 coins)',
    'episode_count': 'Episodes',
    'estimate_hours': 'Half-Life (h)',
    'ci_95_low': 'CI 95% Low',
    'ci_95_high': 'CI 95% High',
    'gap_h1': 'Gap at h=1',
    'gap_h24': 'Gap at h=24',
    'gap_h72': 'Gap at h=72',
    'p_value_beat_null': 'p-value vs Null'
})
display(disp.style.format({
    'Half-Life (h)': '{:.2f}h',
    'CI 95% Low': '{:.2f}h',
    'CI 95% High': '{:.2f}h',
    'Gap at h=1': '{:+.3f}',
    'Gap at h=24': '{:+.3f}',
    'Gap at h=72': '{:+.3f}',
    'p-value vs Null': '{:.4f}'
}))


,Sector (≥3 coins),Episodes,Half-Life (h),CI 95% Low,CI 95% High,Gap at h=1,Gap at h=24,Gap at h=72,p-value vs Null
0,DeFi / oracle,3433,3.38h,2.40h,5.49h,+1.007,+0.385,+0.272,0.0000
1,meme,2168,6.71h,2.86h,7.94h,+1.383,+0.666,+0.365,0.0000
2,payments,4819,3.12h,2.30h,4.50h,+1.255,+0.442,+0.336,0.0000
3,smart-contract platform,9466,3.52h,2.70h,5.09h,+1.078,+0.376,+0.311,0.0000


In [4]:
# Plot excess volatility decay profiles (h = 1..72)
from cryptograph.analysis.persistence import ShockPersistenceAnalyzer

analyzer = ShockPersistenceAnalyzer(horizon_hours=72)
sec_df = pd.read_csv('../data/external/sectors.csv')
eligible_sectors, sec_data = analyzer.extract_sector_episodes(hp, sec_df, min_coins_per_sector=3, seed=42)

hours = np.arange(1, 73)
fig_decay = go.Figure()
colors = {'DeFi / oracle': '#00cc96', 'payments': '#636efa', 'smart-contract platform': '#FFA15A', 'meme': '#FF6692'}

for sec in eligible_sectors:
    d = sec_data[sec]
    sh_m = np.nanmean(d['sh'], axis=0)
    nu_m = np.nanmean(d['nu'], axis=0)
    gap = sh_m - nu_m
    fig_decay.add_trace(go.Scatter(
        x=hours, y=gap, mode='lines',
        name=f"{sec} (gap over null)",
        line=dict(color=colors.get(sec, '#ffffff'), width=2.5)
    ))

fig_decay.add_hline(y=0.0, line_dash='dash', line_color='#888888')
fig_decay.update_layout(
    title='Excess Volatility Profile: E[|r_ds, t+h|] / Baseline - Matched Null (h = 1..72)',
    xaxis_title='Hours post-shock (h)',
    yaxis_title='Excess Volatility Gap',
    height=480
)
fig_decay.show()


[2026-10-05 00:05:13] [INFO] [cryptograph.analysis.persistence] - Eligible sectors (>=3 coins): ['DeFi / oracle', 'meme', 'payments', 'smart-contract platform']


[2026-10-05 00:05:15] [INFO] [cryptograph.analysis.persistence] - Sector 'DeFi / oracle' extracted 3433 shock episodes.


[2026-10-05 00:05:15] [INFO] [cryptograph.analysis.persistence] - Sector 'meme' extracted 2168 shock episodes.


[2026-10-05 00:05:16] [INFO] [cryptograph.analysis.persistence] - Sector 'payments' extracted 4819 shock episodes.


[2026-10-05 00:05:19] [INFO] [cryptograph.analysis.persistence] - Sector 'smart-contract platform' extracted 9466 shock episodes.


## 3. Persistence by Sector, Temporal Split & Crisis Events

### Temporal Split Stability
We evaluate decay half-lives separately across:
- **Train** (2020-01 to 2025-08): includes COVID, China ban, LUNA, Celsius/3AC, FTX, SVB, ETF approval
- **Val** (2025-08 to 2026-03)
- **Test** (2026-03 to 2026-09): strict out-of-sample evaluation period

In [5]:
# Temporal Split Stability Table
split_df = boot_df[boot_df['record_type'] == 'sector_decay_half_life'].copy()
hl_pivot = split_df.pivot(index='sector', columns='split', values='estimate_hours')[['train', 'val', 'test', 'all']]
ep_pivot = split_df.pivot(index='sector', columns='split', values='episode_count')[['train', 'val', 'test', 'all']]
combined = pd.concat([hl_pivot.add_suffix('_hl_hours'), ep_pivot.add_suffix('_episodes')], axis=1)
display(combined.style.format({
    'train_hl_hours': '{:.2f}h',
    'val_hl_hours': '{:.2f}h',
    'test_hl_hours': '{:.2f}h',
    'all_hl_hours': '{:.2f}h',
    'train_episodes': '{:,.0f}',
    'val_episodes': '{:,.0f}',
    'test_episodes': '{:,.0f}',
    'all_episodes': '{:,.0f}'
}))


split,train_hl_hours,val_hl_hours,test_hl_hours,all_hl_hours,train_episodes,val_episodes,test_episodes,all_episodes
sector,,,,,,,,
DeFi / oracle,4.90h,2.26h,3.59h,3.38h,788,675,"1,970","3,433"
meme,7.75h,2.77h,3.79h,6.71h,455,523,"1,190","2,168"
payments,4.69h,2.47h,2.79h,3.12h,"1,657","1,142","2,020","4,819"
smart-contract platform,4.92h,2.56h,3.21h,3.52h,"2,700","2,343","4,423","9,466"


In [6]:
# Historical Crisis Event Study
events_rep = pd.read_csv('../reports/persistence_analysis.csv')
disp_ev = events_rep[[
    'event_name', 'timestamp', 'shock_type', 'blast_radius_pct',
    'propagation_delay_hours', 'mean_persistence_half_life_hours',
    'control_shock_rate', 'event_shock_rate', 'abnormal_shock_rate'
]].rename(columns={
    'event_name': 'Crisis Event',
    'timestamp': 'Trigger Date',
    'shock_type': 'Shock Type',
    'blast_radius_pct': 'Blast Radius',
    'propagation_delay_hours': 'Peak Delay (h)',
    'mean_persistence_half_life_hours': 'Mean Half-Life (h)',
    'control_shock_rate': 'Control Rate',
    'event_shock_rate': 'Event Rate',
    'abnormal_shock_rate': 'Abnormal Rate'
})
display(disp_ev.style.format({
    'Blast Radius': '{:.0%}',
    'Peak Delay (h)': '{:.1f}h',
    'Mean Half-Life (h)': '{:.2f}h',
    'Control Rate': '{:.2%}',
    'Event Rate': '{:.2%}',
    'Abnormal Rate': '{:+.2%}'
}))


,Crisis Event,Trigger Date,Shock Type,Blast Radius,Peak Delay (h),Mean Half-Life (h),Control Rate,Event Rate,Abnormal Rate
0,COVID Black Thursday,2020-03-12 00:00:00,macro_systemic,100%,10.0h,2.00h,5.83%,18.24%,+12.41%
1,China bans financial institutions from crypto services,2021-05-19 00:00:00,macro_systemic,100%,12.0h,2.39h,4.24%,10.39%,+6.14%
2,Terra UST depeg / LUNA collapse,2022-05-09 00:00:00,macro_systemic,100%,16.0h,3.18h,3.42%,20.54%,+17.12%
3,Celsius withdrawal freeze / Three Arrows,2022-06-13 00:00:00,macro_systemic,100%,8.0h,4.38h,7.12%,13.08%,+5.96%
4,FTX collapse,2022-11-08 00:00:00,macro_systemic,100%,17.0h,6.35h,4.46%,15.91%,+11.45%
5,SVB failure / USDC depeg,2023-03-10 00:00:00,macro_systemic,100%,31.0h,2.25h,4.02%,7.99%,+3.97%
6,US spot BTC ETF approval,2024-01-10 00:00:00,macro_systemic,100%,21.0h,3.75h,5.29%,5.95%,+0.66%
7,Yen carry-trade unwind,2024-08-05 00:00:00,macro_systemic,100%,7.0h,2.00h,7.60%,13.01%,+5.41%
8,Oct-10 liquidation cascade,2025-10-10 00:00:00,macro_systemic,100%,20.0h,2.00h,4.23%,10.93%,+6.70%


## 4. Implications for Contagion Models

1. **Resolution of Volatility Artifacts**:
   - Rolling 24h volatility windows previously created artificial predictability: the target reflected 'volatility stays high' rather than genuine contagion.
   - Using deseasonalised returns $r_{\text{ds}}$ without smoothing reveals empirical decay half-lives of **3.1h to 6.7h** (hours, not sub-hours or multi-day window artifacts).
2. **Cross-Sector Heterogeneity**:
   - Meme coins (`DOGE`, `SHIB`, `PEPE`) exhibit prolonged volatility clustering (half-life **6.71h** [95% CI: 2.86h, 7.94h]) compared to payments (**3.12h**) and smart-contract platforms (**3.52h**).
   - All sectors significantly exceed their matched non-shock null ($p < 0.001$), confirming genuine volatility clustering.
3. **Horizon Alignment for Models**:
   - Short horizons ($k = 1\text{h}, 3\text{h}$) capture rapid secondary propagation during the elevated volatility window.
   - Long horizons ($k = 24\text{h}$) capture multi-day contagion when excess volatility has mostly decayed toward baseline ($+0.38$ vs $+1.08$ at $h=1$).
   - Models must be evaluated against honest baselines (e.g. matched nulls and independent cascade) rather than rolling autocorrelations.